# CipherChain — Cryptographic Hash Function Benchmark Analysis

**Purpose:** This notebook ingests empirical benchmark data from the CipherChain test harness and produces
publication-quality figures and summary statistics for an IEEE midsem report on the performance trade-offs
of structurally distinct hash functions.

**Target Algorithms:**

| Algorithm | Construction | Digest (bits) | Design Philosophy |
|-----------|-------------|---------------|-------------------|
| SHA-256 | Merkle-Damgård | 256 | Iterated compression; NIST standard |
| SHA3-256 (Keccak) | Sponge | 256 | Permutation-based; post-quantum hedge |
| Scrypt | Memory-hard PBKDF | 256 | Deliberate memory cost for ASIC resistance |
| BLAKE2b | ARX | 512 | High-speed software hashing via Add-Rotate-XOR |

**Metrics:**
1. Throughput & Latency (MB/s across 1 KB – 100 MB payloads)
2. Memory Footprint (peak RSS delta during hashing)
3. Avalanche Effect (Hamming distance after single-bit input perturbation)

---
## Section 1: Setup & Polars Ingestion

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import polars as pl

# ---------------------------------------------------------------------------
# Matplotlib defaults for IEEE-quality figures
# ---------------------------------------------------------------------------
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "font.size": 11,
        "font.family": "serif",
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

# Colour palette — colourblind-safe, consistent across all plots
ALGO_COLORS: dict[str, str] = {
    "sha256": "#2176AE",  # blue
    "sha3_256": "#57B894",  # teal
    "scrypt": "#FC6136",  # orange
    "blake2b": "#8B5CF6",  # purple
}

ALGO_LABELS: dict[str, str] = {
    "sha256": "SHA-256",
    "sha3_256": "SHA3-256",
    "scrypt": "Scrypt",
    "blake2b": "BLAKE2b",
}

NODE_LABELS: dict[str, str] = {
    "Node A": "Node A (Intel Core Ultra 5 125H / WSL2)",
    "Node B": "Node B (Native Linux Workstation)",
}

ALGO_ORDER: list[str] = ["sha256", "sha3_256", "scrypt", "blake2b"]

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
ROOT = Path("..")  # notebook is in notebooks/
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

print(f"Data directory:    {DATA_DIR}")
print(f"Figures directory: {FIG_DIR}")


In [ ]:
# ---------------------------------------------------------------------------
# Load CSVs into Polars DataFrames
# ---------------------------------------------------------------------------
df_throughput = pl.read_csv(DATA_DIR / "throughput.csv")
df_memory = pl.read_csv(DATA_DIR / "memory.csv")
df_avalanche = pl.read_csv(DATA_DIR / "avalanche.csv")

print("=== Throughput Schema ===")
print(df_throughput.schema)
print(f"Rows: {df_throughput.height}\n")

print("=== Memory Schema ===")
print(df_memory.schema)
print(f"Rows: {df_memory.height}\n")

print("=== Avalanche Schema ===")
print(df_avalanche.schema)
print(f"Rows: {df_avalanche.height}\n")

In [ ]:
# Quick summary statistics for each dataset
print("=== Throughput Summary ===")
display(df_throughput.describe())

print("\n=== Memory Summary ===")
display(df_memory.describe())

print("\n=== Avalanche Summary ===")
display(df_avalanche.describe())

---
## Section 2: Throughput & Latency Analysis

We measure raw hashing speed (MB/s) across dynamically sized payloads ranging from **1 KB to 100 MB**.

### Hardware Profiles Under Test
To eliminate microarchitectural distortion from cross-machine averaging, metrics are isolated by test node:
* **Node A (WSL2 / Hybrid x86_64):** Intel Core Ultra 5 125H (14 cores / 18 threads: 4 P-cores, 8 E-cores, 2 LPE-cores) running inside Ubuntu 24.04 on WSL2.
* **Node B (Bare-Metal Linux):** High-performance native Ubuntu workstation with direct hardware execution, full memory bandwidth, and dedicated cryptographic instruction sets.


In [ ]:
# ---------------------------------------------------------------------------
# Aggregate: mean throughput & latency per (node, algorithm, payload_bytes)
# Isolates hardware profiles to capture microarchitectural variance.
# ---------------------------------------------------------------------------
throughput_agg = (
    df_throughput.group_by(["node", "algorithm", "payload_bytes"])
    .agg(
        pl.col("throughput_mb_per_sec").mean().alias("mean_throughput_mbps"),
        pl.col("throughput_mb_per_sec").std().alias("std_throughput_mbps"),
        pl.col("elapsed_seconds").mean().alias("mean_latency_sec"),
        pl.col("elapsed_seconds").count().alias("n_runs"),
    )
    .with_columns(
        (pl.col("payload_bytes") / 1024).alias("payload_kb"),
        (pl.col("mean_latency_sec") * 1000).alias("mean_latency_ms"),
    )
    .sort(["node", "algorithm", "payload_bytes"])
)

display(throughput_agg)


In [ ]:
# ---------------------------------------------------------------------------
# Plot 1: Throughput vs Payload Size (Separate plots with per-node Y-axis scaling)
# ---------------------------------------------------------------------------
nodes_present = [n for n in ["Node A", "Node B"] if n in throughput_agg["node"].unique().to_list()]
if not nodes_present:
    nodes_present = sorted(throughput_agg["node"].unique().to_list())

for node_id in nodes_present:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    node_df = throughput_agg.filter(pl.col("node") == node_id)

    # Scale Y-axis to each node's peak throughput for clear visual inspection
    node_max_tp = float(node_df["mean_throughput_mbps"].max())
    y_upper = node_max_tp * 1.12

    for algo in ALGO_ORDER:
        subset = node_df.filter(pl.col("algorithm") == algo).sort("payload_bytes")
        if subset.height == 0:
            continue

        x = subset["payload_kb"].to_numpy()
        y = subset["mean_throughput_mbps"].to_numpy()

        ax.plot(
            x,
            y,
            marker="o",
            markersize=6,
            linewidth=2,
            color=ALGO_COLORS[algo],
            label=ALGO_LABELS[algo],
        )

    ax.set_xscale("log")
    ax.set_ylim(0, y_upper)
    ax.set_xlabel("Payload Size (KB, log scale)")
    ax.set_ylabel("Throughput (MB/s)")

    node_desc = NODE_LABELS.get(node_id, node_id)
    ax.set_title(f"Hash Function Throughput vs. Payload Size — {node_desc}")
    ax.legend(frameon=True, fancybox=False, edgecolor="#cccccc")

    # Annotate scrypt cap
    scrypt_1mb = node_df.filter(
        (pl.col("algorithm") == "scrypt") & (pl.col("payload_bytes") == 1024 * 1024)
    )
    if scrypt_1mb.height > 0:
        scrypt_val = float(scrypt_1mb["mean_throughput_mbps"].to_numpy()[0])
        ax.annotate(
            "Scrypt capped at 1 MB\n(cost is parameter-driven)",
            xy=(1024, scrypt_val),
            xytext=(3000, max(scrypt_val * 1.35, y_upper * 0.15)),
            fontsize=9,
            fontstyle="italic",
            color=ALGO_COLORS["scrypt"],
            arrowprops=dict(arrowstyle="->", color=ALGO_COLORS["scrypt"], lw=1.2),
        )

    fig.tight_layout()
    filename = (
        "throughput_node_a.png"
        if "a" in node_id.lower()
        else "throughput_node_b.png"
        if "b" in node_id.lower()
        else f"throughput_{node_id.lower().replace(' ', '_')}.png"
    )
    fig.savefig(FIG_DIR / filename)
    print(f"Saved: {(FIG_DIR / filename).resolve()}")
    plt.show()


### Throughput Observations: Node A vs. Node B Comparison

- **Hardware Acceleration vs. Software Optimization:** SHA-256 achieves the highest throughput on both machines (**787.67 MB/s** on Node A and **2,363.55 MB/s** on Node B), benefiting from dedicated hardware instruction support (SHA-NI).
- **BLAKE2b Software Efficiency:** BLAKE2b is the second fastest algorithm, reaching **316.44 MB/s** on Node A and **1,309.68 MB/s** on Node B, validating its high performance on 64-bit architectures.
- **SHA3-256 (Keccak) Sponge Overhead:** SHA3-256 consistently trails SHA-256 across both machines (**213.39 MB/s** on Node A and **653.28 MB/s** on Node B), illustrating the software computational cost of the 1600-bit permutation state.
- **Scrypt Parameter-Bound Execution:** Scrypt shows low nominal throughput at small payload sizes due to its fixed PBKDF iteration cost, but ramps up as payload size reaches 1 MB (**268.12 MB/s** on Node A and **720.88 MB/s** on Node B).
- **Payload Scaling:** Streaming algorithms reach their peak efficiency once payload sizes exceed 100 KB, maintaining steady throughput across larger buffers as setup overhead is amortized.


---
## Section 3: Memory Footprint & Resource Consumption

We profile peak RSS (Resident Set Size) during hashing to empirically demonstrate the memory cost
differences between algorithms. Scrypt is *designed* to be memory-hard — its internal state
allocates a scratchpad proportional to cost parameters: **Theoretical Memory = (128 * r * N) bytes**.

> **Methodological Note:** At baseline parameters ($N=1024, r=1, p=1$), Scrypt requires only 128 KiB
> ($131{,}072$ bytes), which OpenSSL executes and frees in sub-millisecond time — faster than standard
> OS-level RSS sampling. To empirically capture and demonstrate Scrypt's memory hardness, our harness
> also benchmarks scaled configurations ($N=16384, r=8$ requiring $\approx 16$ MiB, and $N=32768, r=8$
> requiring $\approx 32$ MiB).


In [ ]:
# ---------------------------------------------------------------------------
# Aggregate: peak memory profile per algorithm
# Selects the peak-allocation run for each algorithm so that:
# Peak RSS - Baseline RSS == Peak Memory Delta
# ---------------------------------------------------------------------------
memory_agg = (
    df_memory.sort("delta_memory_mib", descending=True)
    .group_by("algorithm", maintain_order=True)
    .agg(
        pl.col("baseline_memory_mib").first().alias("baseline_mib"),
        pl.col("peak_memory_mib").first().alias("peak_mib"),
        pl.col("delta_memory_mib").first().alias("max_delta_mib"),
        pl.col("delta_memory_mib").mean().alias("mean_delta_mib"),
        pl.col("delta_memory_mib").count().alias("n_runs"),
    )
    .sort("algorithm")
)

display(memory_agg)


In [ ]:
# ---------------------------------------------------------------------------
# Also show per (algorithm, payload_bytes) breakdown
# ---------------------------------------------------------------------------
memory_detail = (
    df_memory.group_by(["algorithm", "payload_bytes"])
    .agg(
        pl.col("baseline_memory_mib").mean().alias("mean_baseline_mib"),
        pl.col("peak_memory_mib").mean().alias("mean_peak_mib"),
        pl.col("delta_memory_mib").mean().alias("mean_delta_mib"),
    )
    .sort(["algorithm", "payload_bytes"])
)

display(memory_detail)

In [ ]:
# ---------------------------------------------------------------------------
# Scrypt Memory-Hardness Scaling: Theoretical vs. Empirical Footprint
# Validates RAM allocation against formula: (128 * r * N) bytes
# ---------------------------------------------------------------------------
scrypt_scaling = (
    df_memory.filter(pl.col("algorithm") == "scrypt")
    .select(
        pl.col("payload_bytes"),
        (pl.col("theoretical_memory_bytes") / (1024 * 1024))
        .round(3)
        .alias("theoretical_ram_mib"),
        pl.col("baseline_memory_mib").round(2).alias("baseline_rss_mib"),
        pl.col("peak_memory_mib").round(2).alias("peak_rss_mib"),
        pl.col("delta_memory_mib").round(4).alias("empirical_delta_mib"),
    )
    .unique()
    .sort("theoretical_ram_mib")
)

display(scrypt_scaling)


In [ ]:
# ---------------------------------------------------------------------------
# Plot 2: Memory Consumption — Grouped Bar Chart (Baseline vs Peak)
# ---------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))

# --- Left panel: Absolute RSS (baseline + peak at peak allocation) ---
algos_sorted = [a for a in ALGO_ORDER if a in memory_agg["algorithm"].to_list()]
mem_data = memory_agg.filter(pl.col("algorithm").is_in(algos_sorted))

# Sort to match ALGO_ORDER using a join with an ordering frame
order_df = pl.DataFrame(
    {
        "algorithm": ALGO_ORDER,
        "_sort": list(range(len(ALGO_ORDER))),
    }
)
mem_data = mem_data.join(order_df, on="algorithm", how="left").sort("_sort")

x_pos = np.arange(len(algos_sorted))
bar_w = 0.35

baselines = mem_data["baseline_mib"].to_numpy()
peaks = mem_data["peak_mib"].to_numpy()

bars_base = ax1.bar(
    x_pos - bar_w / 2,
    baselines,
    width=bar_w,
    label="Baseline RSS",
    color="#94a3b8",
    edgecolor="white",
)
bars_peak = ax1.bar(
    x_pos + bar_w / 2,
    peaks,
    width=bar_w,
    label="Peak RSS",
    color="#ef4444",
    edgecolor="white",
)

max_rss = float(max(baselines.max(), peaks.max()))
ax1.set_ylim(0, max_rss * 1.25)
rss_offset = max_rss * 0.02

for i, (b_val, p_val) in enumerate(zip(baselines, peaks)):
    ax1.text(i - bar_w / 2, b_val + rss_offset, f"{b_val:.1f}", ha="center", va="bottom", fontsize=8.5)
    ax1.text(i + bar_w / 2, p_val + rss_offset, f"{p_val:.1f}", ha="center", va="bottom", fontsize=8.5)

ax1.set_xticks(x_pos)
ax1.set_xticklabels([ALGO_LABELS[a] for a in algos_sorted])
ax1.set_ylabel("Memory (MiB)")
ax1.set_title("Baseline vs. Peak RSS During Hashing", pad=14)
ax1.legend(loc="upper left", frameon=True, fancybox=False, edgecolor="#cccccc")

# --- Right panel: Delta (peak - baseline) ---
bar_colors = [ALGO_COLORS[a] for a in algos_sorted]
deltas = mem_data["max_delta_mib"].to_numpy()

ax2.bar(x_pos, deltas, color=bar_colors, edgecolor="white", width=0.55)

# Expand y-axis limit with generous headroom so bar labels never collide with title
max_d = float(np.max(deltas)) if len(deltas) > 0 and np.max(deltas) > 0 else 1.0
ax2.set_ylim(0, max_d * 1.35)
offset = max(max_d * 0.03, 0.002)

for i, (d, a) in enumerate(zip(deltas, algos_sorted)):
    ax2.text(i, d + offset, f"{d:.4f}", ha="center", va="bottom", fontsize=9)

ax2.set_xticks(x_pos)
ax2.set_xticklabels([ALGO_LABELS[a] for a in algos_sorted])
ax2.set_ylabel("Memory Delta (MiB)")
ax2.set_title("Peak Memory Delta (Peak - Baseline)", pad=14)

fig.suptitle("Memory Footprint Comparison Across Hash Algorithms", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "memory_consumption.png")
print(f"Saved: {(FIG_DIR / 'memory_consumption.png')}")
plt.show()


### Memory Observations

- **SHA-256, SHA3-256, and BLAKE2b** operate almost entirely in CPU registers with negligible heap
  allocation beyond the transient input buffer, yielding near-zero RSS delta.
- **Scrypt Scaled Memory Hardness:** When scaled to $N=16384, r=8$ (theoretical 16 MiB) and $N=32768, r=8$
  (theoretical 32 MiB), Scrypt empirically allocates **15.89 MiB** and **31.92 MiB** of RAM above baseline,
  matching the theoretical $128 \cdot r \cdot N$ formula with >99.7% fidelity.
- This confirms Scrypt's memory hardness: parallelizing attacks requires dedicated hardware RAM proportional
  to the cost parameters, neutralizing ASIC/GPU time-memory trade-offs.


---
## Section 4: Cryptographic Integrity — Avalanche Effect

The **Strict Avalanche Criterion (SAC)** states that flipping any single input bit should cause
each output bit to change with probability 0.5. We test this by:

1. Generating a random payload.
2. Flipping exactly one random bit.
3. Hashing both the original and mutated payloads.
4. Computing the Hamming distance between the two digests.

An ideal hash function yields an **avalanche ratio of approximately 0.50** (50% of output bits change).

In [ ]:
# ---------------------------------------------------------------------------
# Aggregate: avalanche statistics per (algorithm, payload_bytes)
# ---------------------------------------------------------------------------
avalanche_agg = (
    df_avalanche.group_by(["algorithm", "payload_bytes"])
    .agg(
        pl.col("avalanche_ratio").mean().alias("mean_ratio"),
        pl.col("avalanche_ratio").std().alias("std_ratio"),
        pl.col("avalanche_ratio").min().alias("min_ratio"),
        pl.col("avalanche_ratio").max().alias("max_ratio"),
        pl.col("avalanche_ratio").count().alias("n_trials"),
    )
    .sort(["algorithm", "payload_bytes"])
)

display(avalanche_agg)

In [ ]:
# ---------------------------------------------------------------------------
# Per-algorithm summary (all sizes pooled)
# ---------------------------------------------------------------------------
avalanche_by_algo = (
    df_avalanche.group_by("algorithm")
    .agg(
        pl.col("avalanche_ratio").mean().alias("mean_ratio"),
        pl.col("avalanche_ratio").std().alias("std_ratio"),
        pl.col("avalanche_ratio").min().alias("min_ratio"),
        pl.col("avalanche_ratio").max().alias("max_ratio"),
        pl.col("avalanche_ratio").count().alias("n_trials"),
    )
    .sort("algorithm")
)

display(avalanche_by_algo)

In [ ]:
# ---------------------------------------------------------------------------
# Plot 3: Avalanche Distribution — Box Plot
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5.5))

# Prepare data for box plot: list of arrays, one per algorithm
box_data = []
box_labels = []
box_colors = []

for algo in ALGO_ORDER:
    subset = df_avalanche.filter(pl.col("algorithm") == algo)
    if subset.height == 0:
        continue
    box_data.append(subset["avalanche_ratio"].to_numpy())
    box_labels.append(ALGO_LABELS[algo])
    box_colors.append(ALGO_COLORS[algo])

bp = ax.boxplot(
    box_data,
    tick_labels=box_labels,
    patch_artist=True,
    widths=0.5,
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    flierprops=dict(marker="o", markersize=4, alpha=0.5),
)

for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Ideal SAC line at 0.50
ax.axhline(
    y=0.50, color="#dc2626", linestyle="--", linewidth=1.5, label="Ideal SAC (0.50)"
)

# Acceptable band
ax.axhspan(0.45, 0.55, color="#dc2626", alpha=0.06, label="+/-5% tolerance band")

ax.set_ylabel("Avalanche Ratio (Hamming Distance / Total Bits)")
ax.set_title("Avalanche Effect Distribution — Single-Bit-Flip Trials")
ax.legend(loc="lower right", frameon=True, fancybox=False, edgecolor="#cccccc")
ax.set_ylim(0.30, 0.70)

fig.tight_layout()
fig.savefig(FIG_DIR / "avalanche_distribution.png")
print(f"Saved: {(FIG_DIR / 'avalanche_distribution.png')}")
plt.show()

### Avalanche Effect Observations

All four algorithms exhibit mean avalanche ratios tightly clustered around **0.50**, confirming
compliance with the Strict Avalanche Criterion (SAC):

- The interquartile ranges are narrow, indicating low variance across independent single-bit-flip trials.
- No algorithm shows systematic bias toward 0 or 1, as expected for cryptographically secure hash functions.
- The result is independent of payload size — a fundamental property of well-designed hash constructions.

---
## Section 5: Consolidated Summary Table

A single table joining the key metrics across all four algorithms for the IEEE report.

In [ ]:
# ---------------------------------------------------------------------------
# Architecture labels
# ---------------------------------------------------------------------------
ARCHITECTURE: dict[str, str] = {
    "sha256": "Merkle-Damgard",
    "sha3_256": "Sponge (Keccak)",
    "scrypt": "Memory-Hard PBKDF",
    "blake2b": "ARX",
}

# --- Peak throughput per algorithm for Node A and Node B ---
peak_tp_a = (
    df_throughput.filter(pl.col("node") == "Node A")
    .group_by("algorithm")
    .agg(pl.col("throughput_mb_per_sec").max().alias("peak_tp_node_a"))
)

peak_tp_b = (
    df_throughput.filter(pl.col("node") == "Node B")
    .group_by("algorithm")
    .agg(pl.col("throughput_mb_per_sec").max().alias("peak_tp_node_b"))
)

# Robust fallback in case nodes are labeled differently
if peak_tp_a.height == 0 and df_throughput["node"].n_unique() > 0:
    first_node = df_throughput["node"].unique()[0]
    peak_tp_a = (
        df_throughput.filter(pl.col("node") == first_node)
        .group_by("algorithm")
        .agg(pl.col("throughput_mb_per_sec").max().alias("peak_tp_node_a"))
    )

if peak_tp_b.height == 0 and df_throughput["node"].n_unique() > 1:
    second_node = df_throughput["node"].unique()[1]
    peak_tp_b = (
        df_throughput.filter(pl.col("node") == second_node)
        .group_by("algorithm")
        .agg(pl.col("throughput_mb_per_sec").max().alias("peak_tp_node_b"))
    )

# --- Peak memory delta per algorithm ---
peak_memory = df_memory.group_by("algorithm").agg(
    pl.col("delta_memory_mib").max().alias("peak_delta_mib")
)

# --- Mean avalanche ratio per algorithm ---
mean_avalanche = df_avalanche.group_by("algorithm").agg(
    (pl.col("avalanche_ratio").mean() * 100).alias("mean_avalanche_pct"),
)

# --- Build summary table ---
summary = (
    pl.DataFrame(
        {
            "algorithm": ALGO_ORDER,
            "architecture": [ARCHITECTURE[a] for a in ALGO_ORDER],
            "display_name": [ALGO_LABELS[a] for a in ALGO_ORDER],
        }
    )
    .join(peak_tp_a, on="algorithm", how="left")
    .join(peak_tp_b, on="algorithm", how="left")
    .join(peak_memory, on="algorithm", how="left")
    .join(mean_avalanche, on="algorithm", how="left")
    .select(
        pl.col("display_name").alias("Algorithm"),
        pl.col("architecture").alias("Architecture"),
        pl.col("peak_tp_node_a").round(2).alias("Peak Throughput Node A (MB/s)"),
        pl.col("peak_tp_node_b").round(2).alias("Peak Throughput Node B (MB/s)"),
        pl.col("peak_delta_mib").round(4).alias("Peak Memory Delta (MiB)"),
        pl.col("mean_avalanche_pct").round(2).alias("Mean Avalanche (%)"),
    )
)

display(summary)


In [ ]:
# ---------------------------------------------------------------------------
# Export summary to CSV
# ---------------------------------------------------------------------------
summary_path = DATA_DIR / "hashing_summary.csv"
summary.write_csv(summary_path)
print(f"Summary table exported to {summary_path}")

---
## Conclusions

This analysis empirically validates the well-known performance trade-offs across four structurally
distinct hash constructions:

| Finding | Implication |
|---------|-------------|
| SHA-256 delivers the highest throughput | Optimal for bulk data integrity (e.g., Merkle trees, blockchain) |
| SHA3-256 is 2-4x slower in software | Security margin over SHA-256, but at a latency cost |
| Scrypt is deliberately slow and memory-hard | Suitable for password hashing / KDF, not bulk throughput |
| BLAKE2b is fast at small payloads | Strong candidate for application-level checksums and MACs |
| All four satisfy the Strict Avalanche Criterion | Cryptographic integrity confirmed across all constructions |

All benchmark data and figures are reproducible via `uv run python -m benchmarks.run_*` and this notebook.